# ChatDev 运行模式验证 (Snapshot, Replay, Hybrid)
本 Notebook 用于通过独立的实验脚本验证在 `run.py` 和 `camel/model_backend.py` 中重构的三种互斥运行模式代码的正确性。

In [1]:
import os
import glob
import json
import subprocess

def run_command(cmd):
    print(f'Running: {cmd}')
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    return process.returncode

base_task = "Write a python script that prints 'Hello World'"
model = "GPT_4O_MINI"

## 1. 测试 Snapshot 模式
运行指定的任务，并使用 `--snapshot` 保存完整的 API 调用记录 (`api_records.jsonl`)。

In [2]:
project_name_snapshot = 'TestSnapshot'
cmd_snapshot = f'python run.py --snapshot --name {project_name_snapshot} --task "{base_task}" --model {model}'
run_command(cmd_snapshot)

warehouse_dirs = glob.glob(f'WareHouse/{project_name_snapshot}_*')
latest_snapshot_dir = sorted(warehouse_dirs)[-1]
record_file = os.path.join(latest_snapshot_dir, 'api_records.jsonl')

print(f'\nSnapshot directory: {latest_snapshot_dir}')
assert os.path.exists(record_file), 'Snapshot 模式应该生成 api_records.jsonl'

with open(record_file, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f if line.strip()]
print(f'Total API records saved: {len(records)}')


Running: python run.py --snapshot --name TestSnapshot --task "Write a python script that prints 'Hello World'" --model GPT_4O_MINI
**[Preprocessing]**

**ChatDev Starts** (20260406204416)

**Timestamp**: 20260406204416

**config_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\ChatChainConfig.json

**config_phase_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\PhaseConfig.json

**config_role_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\RoleConfig.json

**task_prompt**: Write a python script that prints 'Hello World'

**project_name**: TestSnapshot

**Log File**: d:\Works\code\winter-like-ai\ChatDev\WareHouse\TestSnapshot_DefaultOrganization_20260406204416.log

**ChatDevConfig**:
ChatEnvConfig.with_memory: False
ChatEnvConfig.clear_structure: True
ChatEnvConfig.git_management: False
ChatEnvConfig.gui_design: True
ChatEnvConfig.incremental_develop: False
ChatEnvConfig.background_prompt: ChatDev is a software company powered by mu

## 2. 测试 Replay 模式
使用上一步 Snapshot 生成的 `api_records.jsonl`，执行纯回放的透明代理调用。按照逻辑，它不应生成新的 `api_records.jsonl`，内部也不会涉及 Git 追踪，仅代理模型的请求输出。

In [3]:
project_name_replay = 'TestReplay'
cmd_replay = f'python run.py --replay "{record_file}" --name {project_name_replay} --task "{base_task}" --model {model}'
run_command(cmd_replay)

warehouse_dirs = glob.glob(f'WareHouse/{project_name_replay}_*')
latest_replay_dir = sorted(warehouse_dirs)[-1]
replay_record_file = os.path.join(latest_replay_dir, 'api_records.jsonl')

print(f'\nReplay directory: {latest_replay_dir}')
assert not os.path.exists(replay_record_file), 'Replay 模式作为透明代理，不应该再次输出 api_records.jsonl'
print('Replay execution validated successfully.')


Running: python run.py --replay "WareHouse\TestSnapshot_DefaultOrganization_20260406204416\api_records.jsonl" --name TestReplay --task "Write a python script that prints 'Hello World'" --model GPT_4O_MINI
**[Preprocessing]**

**ChatDev Starts** (20260406204846)

**Timestamp**: 20260406204846

**config_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\ChatChainConfig.json

**config_phase_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\PhaseConfig.json

**config_role_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\RoleConfig.json

**task_prompt**: Write a python script that prints 'Hello World'

**project_name**: TestReplay

**Log File**: d:\Works\code\winter-like-ai\ChatDev\WareHouse\TestReplay_DefaultOrganization_20260406204846.log

**ChatDevConfig**:
ChatEnvConfig.with_memory: False
ChatEnvConfig.clear_structure: True
ChatEnvConfig.git_management: False
ChatEnvConfig.gui_design: True
ChatEnvConfig.incremental_develop: False
ChatEn

## 3. 测试 Hybrid 模式
结合 Replay 和 Snapshot。设定在中间某个节点切换为实时调用（例如从第 2 个节点开始切换，基于 1-based 索引）。它既能代理早期节点的表现，同时也应生成新的包含全流程节点的完整 `api_records.jsonl` 文件。

In [4]:
project_name_hybrid = 'TestHybrid'
hybrid_node = 2
cmd_hybrid = f'python run.py --hybrid "{record_file}" --hybrid-node {hybrid_node} --name {project_name_hybrid} --task "{base_task}" --model {model}'
run_command(cmd_hybrid)

warehouse_dirs = glob.glob(f'WareHouse/{project_name_hybrid}_*')
latest_hybrid_dir = sorted(warehouse_dirs)[-1]
hybrid_record_file = os.path.join(latest_hybrid_dir, 'api_records.jsonl')

print(f'\nHybrid directory: {latest_hybrid_dir}')
assert os.path.exists(hybrid_record_file), 'Hybrid 模式应该生成包含全程代理和实时调用的 api_records.jsonl'

with open(hybrid_record_file, 'r', encoding='utf-8') as f:
    hybrid_records = [json.loads(line) for line in f if line.strip()]
print(f'Total API records saved in Hybrid mode: {len(hybrid_records)}')


Running: python run.py --hybrid "WareHouse\TestSnapshot_DefaultOrganization_20260406204416\api_records.jsonl" --hybrid-node 2 --name TestHybrid --task "Write a python script that prints 'Hello World'" --model GPT_4O_MINI
**[Preprocessing]**

**ChatDev Starts** (20260406205215)

**Timestamp**: 20260406205215

**config_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\ChatChainConfig.json

**config_phase_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\PhaseConfig.json

**config_role_path**: d:\Works\code\winter-like-ai\ChatDev\CompanyConfig\Default\RoleConfig.json

**task_prompt**: Write a python script that prints 'Hello World'

**project_name**: TestHybrid

**Log File**: d:\Works\code\winter-like-ai\ChatDev\WareHouse\TestHybrid_DefaultOrganization_20260406205215.log

**ChatDevConfig**:
ChatEnvConfig.with_memory: False
ChatEnvConfig.clear_structure: True
ChatEnvConfig.git_management: False
ChatEnvConfig.gui_design: True
ChatEnvConfig.incremental_devel